# 🎬 Movie Genre Classification
**Goal:** Predict a movie's genre from its plot description using machine learning.

**Big idea:** If a description mentions 'spaceship' and 'aliens' → probably Sci-Fi. Can a model learn this automatically?

### Steps:
1. Load the data
2. Clean the text
3. Convert text → numbers (TF-IDF)
4. Train & compare 3 models
5. Make predictions

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print('Libraries loaded!')

## Step 2: Load & Explore the Data

In [ ]:
# Load the dataset (columns are separated by :::)
df = pd.read_csv('train_data.txt', sep=':::', engine='python', header=None)
df.columns = ['id', 'name', 'genre', 'description']

print(f'Total movies: {len(df)}')
print(f'Unique genres: {df["genre"].nunique()}')
df.head(3)

In [ ]:
# How many movies per genre? (top 10)
genre_counts = df['genre'].str.strip().value_counts().head(10)
print(genre_counts)

# Quick bar chart
genre_counts.plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.title('Top 10 Genres')
plt.xlabel('Number of Movies')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# NOTE: The dataset is imbalanced — Drama and Documentary dominate!

## Step 3: Clean the Text
We need to standardize the text before feeding it to a model.

In [ ]:
def clean_text(text):
    """Clean a movie description."""
    text = str(text).lower()              # lowercase
    text = re.sub(r'<.*?>', ' ', text)    # remove HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text) # keep only letters
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces
    return text

# Apply to every description
df['clean_description'] = df['description'].apply(clean_text)
df['genre'] = df['genre'].str.strip().str.lower()

# Before vs after
print('BEFORE:', df['description'].iloc[0][:100])
print()
print('AFTER: ', df['clean_description'].iloc[0][:100])

## Step 4: Split Data into Train & Validation Sets
We train on 80% of data and test on the remaining 20%.

In [ ]:
X = df['clean_description']  # input features
y = df['genre']              # labels we want to predict

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,    # 20% for validation
    random_state=42,  # for reproducibility
    stratify=y        # keep genre proportions the same in both splits
)

print(f'Training samples:   {len(X_train)}')
print(f'Validation samples: {len(X_val)}')

## Step 5: Convert Text → Numbers with TF-IDF

Machines can't read words — we convert text to a numeric matrix.

- **TF (Term Frequency):** How often does a word appear in *this* description?
- **IDF (Inverse Document Frequency):** How rare is that word across *all* descriptions?
- Common words like 'the', 'a', 'is' get low scores. Rare/specific words get high scores.

In [ ]:
# Create the vectorizer
tfidf = TfidfVectorizer(
    max_features=50000,   # keep only the 50k most useful words
    ngram_range=(1, 2),   # use single words AND word pairs (e.g. 'serial killer')
    stop_words='english'  # ignore very common words: 'the', 'a', 'in', ...
)

# IMPORTANT: fit only on training data to avoid 'data leakage'
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)  # just transform — don't refit!

print(f'Training matrix shape:   {X_train_tfidf.shape}')
print(f'Validation matrix shape: {X_val_tfidf.shape}')
# Rows = movies, Columns = word/phrase features

## Step 6: Train & Compare 3 Models

| Model | Description |
|---|---|
| **Logistic Regression** | A strong, reliable baseline for classification |
| **Naive Bayes** | Very fast; works well with word counts |
| **Linear SVM** | Often best for high-dimensional text data |

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=5),
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Linear SVM':          LinearSVC(C=1.0, max_iter=2000)
}

results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)           # train
    preds = model.predict(X_val_tfidf)          # predict on validation set
    acc = accuracy_score(y_val, preds)          # measure accuracy
    results[name] = {'model': model, 'accuracy': acc, 'preds': preds}
    print(f'{name}: {acc:.2%}')

In [ ]:
# Pick the best model automatically
best_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_name]['model']
best_preds = results[best_name]['preds']

print(f'Best model: {best_name} ({results[best_name]["accuracy"]:.2%})')

# Bar chart comparison
names = list(results.keys())
accs  = [results[n]['accuracy'] * 100 for n in names]

plt.figure(figsize=(7, 4))
bars = plt.bar(names, accs, color=['green' if n == best_name else 'steelblue' for n in names])
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{acc:.1f}%', ha='center')
plt.ylim(0, 100)
plt.ylabel('Validation Accuracy (%)')
plt.title('Model Comparison')
plt.tight_layout()
plt.show()

In [ ]:
# Detailed breakdown: precision, recall, F1 per genre
print(f'=== {best_name} — Detailed Report ===')
print(classification_report(y_val, best_preds, zero_division=0))

## Step 7: Predict on New Test Data

In [ ]:
# Load test data (no genre column — that's what we predict)
df_test = pd.read_csv('test_data.txt', sep=':::', engine='python', header=None)
df_test.columns = ['id', 'name', 'description']

# Clean and vectorize (using the SAME tfidf fitted on training data)
df_test['clean_description'] = df_test['description'].apply(clean_text)
X_test_tfidf = tfidf.transform(df_test['clean_description'])

# Predict!
df_test['predicted_genre'] = best_model.predict(X_test_tfidf)

print(f'Predictions made for {len(df_test)} movies.')
df_test[['id', 'name', 'predicted_genre']].head(10)

In [ ]:
# Save predictions to a CSV
df_test[['id', 'name', 'predicted_genre']].to_csv('predictions.csv', index=False)
print('Saved to predictions.csv')

## Bonus: Try It Yourself!
Type any movie description and see what genre the model predicts.

In [ ]:
def predict_genre(description):
    cleaned  = clean_text(description)
    vector   = tfidf.transform([cleaned])
    genre    = best_model.predict(vector)[0]
    return genre

# Try your own descriptions!
test_descriptions = [
    "A young wizard discovers he has magical powers and goes to a school for witchcraft.",
    "Two detectives hunt down a serial killer who uses the seven deadly sins.",
    "An astronaut gets stranded on Mars and must survive using science.",
]

for desc in test_descriptions:
    genre = predict_genre(desc)
    print(f'Description: {desc[:60]}...')
    print(f'Predicted:   {genre.upper()}')
    print()

---
## Summary

```
Raw text  →  Clean text  →  TF-IDF numbers  →  Train model  →  Predict genre
```

**What worked well:**
- TF-IDF with bigrams captures phrases like 'serial killer' or 'outer space'
- Linear models are fast and surprisingly effective on text

**What could be improved:**
- Dataset is imbalanced (Drama dominates) → try class weights
- Try BERT or other transformer models for much higher accuracy